# Dear Canada — Corpus-wide Topic Model (shared themes & events)

**Question this notebook answers:** *which themes and events recur across the Dear Canada books, and which belong to only one book or one moment in history?*

**Input:** every CSV in `./data/` with columns `book, chapter_id, entry_id, date, entry_text` (the same format as the KG pipeline). Text is assumed to be raw and uncleaned.

**Method in one paragraph.** Clean the text → lemmatise and keep content words, dropping people's names (spaCy) → merge short consecutive diary entries into ~150-word *segments* → build TF-IDF, keeping only vocabulary that appears in **several different books** → fit an NMF topic model, choosing the number of topics by NPMI coherence → score every entry → measure, per topic, how many books it is prominent in and how tightly it clusters in historical time.

**Why that last measurement matters.** A topic that is prominent in many books *and* spread across centuries is a **shared theme** (e.g. family separation). A topic that is prominent in several books but packed into a narrow span of years is more likely a **shared historical event or period** (e.g. an epidemic, a war). A topic carried by one book is **book-specific**. These categories are heuristics to guide your reading, not findings; the exemplar passages are there so you can check them.

**Outputs** (in `./output/topics_corpus/`): `topic_summary.csv`, `topic_terms.csv`, `book_topic_matrix.csv`, `topic_period_matrix.csv`, `exemplars.csv`, `entry_topics.csv` (joins to the KG on `entry_id`), and interactive HTML figures in `figures/`.

**Requires:** `pip install pandas numpy scipy scikit-learn spacy plotly tqdm` and `python -m spacy download en_core_web_sm`. The optional Claude labelling step also needs `anthropic` and `python-dotenv`.

## 0. Setup & configuration

In [4]:
import os
import re
import json
import html
import hashlib
import unicodedata
import warnings
from math import ceil
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import sparse
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import pdist, jensenshannon
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [13]:
# --- paths -------------------------------------------------------------------
DATA_DIR = Path("./data")
CSV_GLOB = "*_entries.csv"                         
OUT_DIR = Path("./output/topics_corpus")
FIG_DIR = OUT_DIR / "figures"
CACHE_DIR = OUT_DIR / "cache"
for d in (OUT_DIR, FIG_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEED = 42

# --- text preprocessing ------------------------------------------------------
SPACY_MODEL = "en_core_web_sm"              # sm is plenty here; en_core_web_trf is far slower for no topic-model gain
REBUILD_LEMMA_CACHE = False
EXTRA_STOPWORDS = set()                     # add corpus-specific junk you spot in the topics, e.g. {"chapter", "ship"}

# --- dates -------------------------------------------------------------------
# Only needed for books where NO entry's date contains a year, e.g. {"SofS_entries_halved": 1847}
BOOK_START_YEAR = {'FitS': 1815, 'PasWide': 1926}
BOOK_TITLES = {}                            # optional display names, e.g. {"SofS_entries_halved": "A Sea of Sorrows"}

# --- segmenting & vocabulary -------------------------------------------------
MIN_SEGMENT_TOKENS = 150                    # content words per segment; lower = more, noisier segments
NGRAM_RANGE = (1, 2)
MIN_DF = 5                                  # a term must appear in >= this many segments overall
MAX_DF = 0.40                               # ...and in fewer than this share of segments (drops near-universal words)
MIN_BOOK_FRACTION = 0.08                    # KEY KNOB: a term must be used in >= this share of books (min 2 books)...
MIN_SEGS_PER_BOOK = 3                       # ...in at least this many segments of each such book.
                                            # Raising MIN_BOOK_FRACTION -> broader, more universal themes.
                                            # Lowering it -> lets in vocabulary of events covered by only 2-3 books.

# --- topic model -------------------------------------------------------------
K_RANGE = range(8, 41, 4)                   # candidate numbers of topics for the coherence sweep
K_FINAL = None                              # None = pick best coherence; or set an int after looking at the sweep plot
COHERENCE_TOPN = 10
MIN_SEGS_PER_TOPIC = 4                      # caps K so each topic has >= this many segments of data

# --- shared / event heuristics -----------------------------------------------
PROMINENCE_FACTOR = 1.0                     # topic is "prominent" in a book if its share >= factor / K (i.e. at least average)
SHARED_THEME_MIN_BOOK_FRAC = 0.30           # prominent in >= 30% of books -> candidate shared theme
EVENT_MAX_IQR_YEARS = 8                     # interquartile span of years <= this -> candidate event / period
BIN_YEARS = 10                              # width of historical-period bins for the period heatmap

# --- optional Claude labelling -----------------------------------------------
USE_LLM_LABELS = False                      # needs ANTHROPIC_API_KEY (loaded from .env like the KG notebook)
MODEL_NAME = "claude-sonnet-5-5"            # match whatever MODEL_NAME your KG notebook uses

## 1. Load every book

Reads all CSVs in `DATA_DIR`, checks the schema, de-duplicates on `book + entry_id`, and puts entries back in reading order (chapter number, then entry number, parsed the same way as `load_book` in the KG notebook).

In [14]:
REQUIRED = ["book", "chapter_id", "entry_id", "date", "entry_text"]

def read_csv_robust(path):
    for enc in ("utf-8-sig", "utf-8", "cp1252", "latin-1"):
        try:
            return pd.read_csv(path, dtype=str, encoding=enc, keep_default_na=False)
        except UnicodeDecodeError:
            continue
    raise RuntimeError(f"Could not decode {path}")

def load_corpus(data_dir, pattern):
    files = sorted(data_dir.glob(pattern))
    if not files:
        raise FileNotFoundError(f"No files matching {pattern!r} in {data_dir.resolve()}")
    frames = []
    for path in files:
        df = read_csv_robust(path)
        missing = set(REQUIRED) - set(df.columns)
        if missing:
            print(f"  skipped {path.name}: missing columns {sorted(missing)}")
            continue
        df = df[REQUIRED].copy()
        df["book"] = df["book"].str.strip().replace("", path.stem)
        df["source_file"] = path.name
        frames.append(df)
    corpus = pd.concat(frames, ignore_index=True)
    corpus["file_order"] = np.arange(len(corpus))
    corpus["uid"] = corpus["book"] + "|" + corpus["entry_id"]
    corpus["_ch"] = pd.to_numeric(corpus["chapter_id"].str.extract(r"(\d+)")[0], errors="coerce").fillna(0)
    corpus["_en"] = pd.to_numeric(corpus["entry_id"].str.extract(r"e(\d+)$")[0], errors="coerce")
    corpus["_en"] = corpus["_en"].fillna(corpus["file_order"])
    n0 = len(corpus)
    corpus = corpus.drop_duplicates("uid", keep="first")
    if len(corpus) < n0:
        print(f"  dropped {n0 - len(corpus)} duplicate book+entry_id rows")
    corpus = corpus.sort_values(["book", "_ch", "_en", "file_order"]).reset_index(drop=True)
    corpus["pos"] = corpus.groupby("book").cumcount()
    return corpus

corpus = load_corpus(DATA_DIR, CSV_GLOB)
print(f"{len(corpus):,} entries from {corpus['book'].nunique()} books")

3,266 entries from 20 books


## 2. Clean the raw text

In [15]:
# Raw CSV text is NOT assumed to be clean. This function removes the usual
# artefacts of scanned / converted book text *before* any linguistic processing.
# It is deliberately conservative: it never deletes words, only formatting noise.

QUOTES = str.maketrans({
    "\u2018": "'", "\u2019": "'", "\u201a": "'", "\u201b": "'",
    "\u201c": '"', "\u201d": '"', "\u2013": "-", "\u2014": " - ",
})

def clean_text(text, date_raw=""):
    if not isinstance(text, str) or not text.strip():
        return ""
    t = html.unescape(text)
    t = re.sub(r"<[^>]+>", " ", t)                          # stray HTML / XML tags
    t = unicodedata.normalize("NFKC", t)                    # ligatures (ﬁ -> fi), odd spaces
    t = t.replace("\u00ad", "")                             # soft hyphens
    t = re.sub(r"https?://\S+|www\.\S+", " ", t)            # URLs
    t = re.sub(r"(\w)-[ \t]*\r?\n[ \t]*(\w)", r"\1\2", t)   # words hyphenated across line breaks
    t = t.translate(QUOTES)                                 # curly quotes / dashes -> ASCII
    t = re.sub(r"\[\d{1,3}\]", " ", t)                      # footnote markers like [12]
    t = re.sub(r"(?im)^[ \t]*(page[ \t]+)?\d{1,4}[ \t]*$", " ", t)   # lines that are only a page number
    t = re.sub(r"\s+", " ", t).strip()                      # collapse all whitespace
    d = str(date_raw).strip()
    if d and t.lower().startswith(d.lower()):               # date heading repeated inside the text
        t = t[len(d):].lstrip(" .,:;-")
    return t

In [20]:
corpus["text_clean"] = [clean_text(t, d) for t, d in zip(corpus["entry_text"], corpus["date"])]
empty = corpus["text_clean"].str.len() == 0
if empty.any():
    print(f"Dropping {int(empty.sum())} entries with no text after cleaning")
    corpus = corpus[~empty].reset_index(drop=True)
    corpus["pos"] = corpus.groupby("book").cumcount()

# Eyeball one before/after to confirm the cleaning is doing what you expect
i = corpus.index[len(corpus) // 2]
print("RAW  :", repr(corpus.loc[i, "entry_text"][:300]))
print("CLEAN:", repr(corpus.loc[i, "text_clean"][:300]))
corpus.head(3)

Dropping 47 entries with no text after cleaning
RAW  : 'Alice tells me that the Indians have been very kind to the settlers since they have been here. She has begun to spend time with me and White Loon, trying to learn a little of their language as well. Kate stays well away and makes endless comments when we are back in our hut, about fraternizing with '
CLEAN: 'Alice tells me that the Indians have been very kind to the settlers since they have been here. She has begun to spend time with me and White Loon, trying to learn a little of their language as well. Kate stays well away and makes endless comments when we are back in our hut, about fraternizing with '


,book,chapter_id,entry_id,date,entry_text,source_file,file_order,uid,_ch,_en,pos,year,month,day,year_inferred,text_clean
0,AToBD,01,ch01_e1,"May 15, 1862","Upper Fort Garry, Red River Settlement We buri...",AToBD_entries.csv,0,AToBD|ch01_e1,1,1,0,1862,5.0,15.0,False,"Upper Fort Garry, Red River Settlement We buri..."
1,AToBD,01,ch01_e2,"May 15, 1862",", evening I finally believe Mama is never comi...",AToBD_entries.csv,1,AToBD|ch01_e2,1,2,1,1862,5.0,15.0,False,", evening I finally believe Mama is never comi..."
2,AToBD,01,ch01_e3,"May 17, 1862",Our situation is not good. We have spent all o...,AToBD_entries.csv,2,AToBD|ch01_e3,1,3,2,1862,5.0,17.0,False,Our situation is not good. We have spent all o...


## 3. Dates → years

The period analysis needs a year per entry. See the comments in the cell for exactly how missing years are inferred; inferred years are flagged in `year_inferred`.

In [16]:
# Dates are messy in diary books ("Tuesday, May 4" has no year). Strategy:
#   1. read year / month / day out of each raw date string (ISO or free text);
#   2. carry the most recent explicit year forward within a book;
#   3. if the month number wraps backwards by 6+ months (Dec -> Jan), add a year;
#   4. flag every year that was inferred rather than read, so you can audit it.
# If a book has no year anywhere, add it to BOOK_START_YEAR in the config cell.

MONTH_NAMES = {
    "january": 1, "february": 2, "march": 3, "april": 4, "may": 5, "june": 6,
    "july": 7, "august": 8, "september": 9, "october": 10, "november": 11, "december": 12,
    "jan": 1, "feb": 2, "mar": 3, "apr": 4, "jun": 6, "jul": 7, "aug": 8,
    "sep": 9, "sept": 9, "oct": 10, "nov": 11, "dec": 12,
}
_month_re = re.compile(r"\b(" + "|".join(sorted(MONTH_NAMES, key=len, reverse=True)) + r")\b\.?", re.I)
_year_re = re.compile(r"\b(1[5-9]\d{2}|20[0-2]\d)\b")
_iso_re = re.compile(r"\b(\d{4})-(\d{1,2})(?:-(\d{1,2}))?\b")

def parse_date_parts(raw):
    s = str(raw or "").strip()
    if not s:
        return (None, None, None)
    m = _iso_re.search(s)
    if m and 1 <= int(m.group(2)) <= 12:
        return int(m.group(1)), int(m.group(2)), (int(m.group(3)) if m.group(3) else None)
    y = _year_re.search(s)
    mo = _month_re.search(s)
    year = int(y.group(1)) if y else None
    month = MONTH_NAMES[mo.group(1).lower()] if mo else None
    day = None
    rest = _year_re.sub(" ", s)
    dm = re.search(r"\b(\d{1,2})(?:st|nd|rd|th)?\b", rest)
    if dm and 1 <= int(dm.group(1)) <= 31:
        day = int(dm.group(1))
    return year, month, day

def resolve_book_dates(g, start_year=None):
    parts = [parse_date_parts(x) for x in g["date"]]
    explicit = [p[0] for p in parts if p[0]]
    cur = explicit[0] if explicit else start_year
    years, inferred, prev_month = [], [], None
    for y, m, d in parts:
        if y:
            cur, flag = y, False
        else:
            flag = cur is not None
            if cur is not None and m and prev_month and m < prev_month and (prev_month - m) >= 6:
                cur += 1
        if m:
            prev_month = m
        years.append(cur if cur is not None else np.nan)
        inferred.append(flag)
    return pd.DataFrame({
        "year": years,
        "month": pd.to_numeric([p[1] for p in parts], errors="coerce"),
        "day": pd.to_numeric([p[2] for p in parts], errors="coerce"),
        "year_inferred": inferred,
    }, index=g.index)

In [17]:
date_parts = pd.concat([resolve_book_dates(g, BOOK_START_YEAR.get(b)) for b, g in corpus.groupby("book", sort=False)])
corpus = corpus.join(date_parts)

book_overview = corpus.groupby("book").agg(
    entries=("uid", "size"),
    year_min=("year", "min"), year_median=("year", "median"), year_max=("year", "max"),
    pct_year_inferred=("year_inferred", "mean"),
).round(2)
no_year = book_overview.index[book_overview["year_median"].isna()].tolist()
if no_year:
    print("!! No year could be read for:", no_year)
    print("   Add them to BOOK_START_YEAR in the config cell. Until then they are left out of the period analysis.")
book_overview

,entries,year_min,year_median,year_max,pct_year_inferred
book,,,,,
AToBD,114,1862,1862.0,1862,0.01
AinUTL,167,1666,1666.0,1667,0.00
BFfH,134,1916,1917.0,1918,0.96
BUoL,129,1884,1885.0,1885,0.00
CofOO,258,1812,1866.0,2013,0.01
DRtoF,201,1863,1863.0,1864,0.00
DofTT,169,1887,1887.0,1888,0.99
DofmyC,175,1759,1759.0,1759,0.00
EftW,259,1940,1940.0,1941,0.00


## 4. Lemmatise & filter to content words

This is the slow step (spaCy over every entry). Results are cached in `output/topics_corpus/cache/`, so adding a book later only processes the new entries.

In [22]:
# Linguistic preprocessing: lemmatise, keep content words, drop people's names.
#
# Why names are removed: in a diary novel the most frequent "words" are the
# narrator's family. Left in, every topic becomes "a book's cast list" rather
# than a theme. Place names and group names (GPE/LOC/FAC/NORP/EVENT) ARE kept,
# because they carry the spatial and cultural/ethnic signal this project cares about.

KEEP_POS = {"NOUN", "VERB", "ADJ"}
KEEP_ENTS = {"GPE", "LOC", "FAC", "NORP", "EVENT"}

MONTHS_DAYS = set("""january february march april may june july august september october november
december jan feb mar apr jun jul aug sep sept oct nov dec monday tuesday wednesday thursday
friday saturday sunday mon tue tues wed thu thur thurs fri sat sun""".split())

GENERIC = set("""dear diary today tomorrow yesterday tonight morning evening afternoon night day
week month year time thing way say go come get make know think see look want take give tell ask
let put seem feel keep try find little long good great old new big first last next other own able
sure right oh yes well mr mrs miss ms sir madam like just really very much many one two three
also even still back around away again always never ever already though""".split())

STOPWORDS = MONTHS_DAYS | GENERIC | set(EXTRA_STOPWORDS)

def load_nlp(name):
    try:
        import spacy
        nlp = spacy.load(name, disable=["parser"])
        nlp.max_length = 2_000_000
        print(f"Loaded spaCy model: {name}")
        return nlp
    except Exception as exc:
        print(f"!! Could not load spaCy model {name!r} ({exc}).")
        print("   Falling back to plain regex tokenisation: no lemmas, no name removal.")
        print(f"   Fix with:  python -m spacy download {name}")
        return None

def lemmatise_texts(texts, nlp, batch_size=32, n_process=1):
    out = []
    if nlp is None:
        for t in texts:
            toks = [w for w in re.findall(r"[a-z]{3,}", t.lower()) if w not in STOPWORDS]
            out.append(" ".join(toks))
        return out
    for doc in tqdm(nlp.pipe(texts, batch_size=batch_size, n_process=n_process), total=len(texts), desc="spaCy"):
        toks = []
        for tok in doc:
            if tok.ent_type_ == "PERSON":
                continue
            if not (tok.pos_ in KEEP_POS or tok.ent_type_ in KEEP_ENTS):
                continue
            if not tok.is_alpha or tok.is_stop:
                continue
            lem = tok.lemma_.lower()
            if len(lem) < 3 or lem in STOPWORDS:
                continue
            toks.append(lem)
        out.append(" ".join(toks))
    return out

def add_lemmas(df, nlp, cache_path, rebuild=False):
    # Incremental cache: only entries whose cleaned text is new or changed are re-processed,
    # so adding a book later does not re-run spaCy over the whole corpus.
    df = df.copy()
    df["text_hash"] = df["text_clean"].map(lambda s: hashlib.md5(s.encode("utf-8")).hexdigest()[:12])
    if cache_path.exists() and not rebuild:
        cache = pd.read_csv(cache_path, dtype=str, keep_default_na=False)
        df = df.merge(cache[["uid", "text_hash", "lemmas"]], on=["uid", "text_hash"], how="left")
    else:
        df["lemmas"] = pd.Series([None] * len(df), index=df.index, dtype="object")
    df["lemmas"] = df["lemmas"].astype("object")
    todo = df["lemmas"].isna()
    print(f"{int((~todo).sum()):,} entries from cache, {int(todo.sum()):,} to process")
    if todo.any():
        df.loc[todo, "lemmas"] = lemmatise_texts(df.loc[todo, "text_clean"].tolist(), nlp)
        df[["uid", "text_hash", "lemmas"]].to_csv(cache_path, index=False)
    df["lemmas"] = df["lemmas"].fillna("")
    df["n_tokens"] = df["lemmas"].str.split().str.len()
    return df

In [23]:
nlp = load_nlp(SPACY_MODEL)
corpus = add_lemmas(corpus, nlp, CACHE_DIR / f"lemmas_{SPACY_MODEL}.csv", rebuild=REBUILD_LEMMA_CACHE)
print(f"Median content words per entry: {int(corpus['n_tokens'].median())}")
corpus[["book", "entry_id", "n_tokens", "lemmas"]].head(3)

Loaded spaCy model: en_core_web_sm
0 entries from cache, 3,219 to process


spaCy:   0%|          | 0/3219 [00:00<?, ?it/s]

Median content words per entry: 38


,book,entry_id,n_tokens,lemmas
0,AToBD,ch01_e1,24,bury mean write word dead table head bent pen ...
1,AToBD,ch01_e2,73,believe stand burying coffin small grave baby ...
2,AToBD,ch01_e3,45,situation spend money return gold field rich r...


## 5. Build segments

In [24]:
# Diary entries are often 50-200 words, which is too short for stable topic estimates.
# So models are FIT on "segments": consecutive entries (never crossing a chapter or book
# boundary) merged until they reach MIN_SEGMENT_TOKENS content words. Every individual
# entry is then scored against the fitted model afterwards, so you still get per-entry
# topic weights that join back to your knowledge graph on entry_id.

def build_segments(df, min_tokens):
    segs = []
    for (book, chap), g in df.groupby(["book", "chapter_id"], sort=False):
        cur, n = [], 0
        for r in g.itertuples():
            cur.append(r); n += r.n_tokens
            if n >= min_tokens:
                segs.append(cur); cur, n = [], 0
        if cur:
            if segs and segs[-1][0].book == book and segs[-1][0].chapter_id == chap:
                segs[-1].extend(cur)          # fold a short tail into the previous segment
            else:
                segs.append(cur)              # chapter too short to reach the threshold
    rows = []
    for i, s in enumerate(segs):
        yrs = [r.year for r in s if not pd.isna(r.year)]
        rows.append({
            "seg_id": i,
            "book": s[0].book,
            "chapter_id": s[0].chapter_id,
            "first_pos": s[0].pos,
            "uids": [r.uid for r in s],
            "n_entries": len(s),
            "n_tokens": sum(r.n_tokens for r in s),
            "year": float(np.median(yrs)) if yrs else np.nan,
            "lemmas": " ".join(r.lemmas for r in s),
            "text_clean": " ".join(r.text_clean for r in s),
        })
    return pd.DataFrame(rows)

In [25]:
seg_df = build_segments(corpus, MIN_SEGMENT_TOKENS)
print(f"{len(seg_df):,} segments (median {int(seg_df['n_tokens'].median())} content words, "
      f"{seg_df['n_entries'].median():.0f} entries each)")
seg_df.groupby("book").size().describe()

856 segments (median 195 content words, 3 entries each)


count    20.000000
mean     42.800000
std       6.212043
min      33.000000
25%      38.750000
50%      43.000000
75%      45.250000
max      55.000000
dtype: float64

## 6. Vectorise, keeping only vocabulary shared across books

TF-IDF over unigrams and bigrams, then the step that makes this a *cross-book* model: any term used in fewer than `MIN_BOOKS` books is discarded. Without it, each book's own place names and rare nouns form topics that only describe that one book.

In [26]:
n_books = seg_df["book"].nunique()
MIN_BOOKS = 1 if n_books < 2 else max(2, int(round(MIN_BOOK_FRACTION * n_books)))
print(f"{n_books} books -> a term must appear in >= {MIN_BOOKS} books (>= {MIN_SEGS_PER_BOOK} segments each)")

vec = TfidfVectorizer(lowercase=False, tokenizer=str.split, token_pattern=None, preprocessor=None,
                      ngram_range=NGRAM_RANGE, min_df=MIN_DF, max_df=MAX_DF, sublinear_tf=True)
X_all = vec.fit_transform(seg_df["lemmas"])
vocab_all = np.array(vec.get_feature_names_out())

book_codes, b_idx_all = np.unique(seg_df["book"].values, return_inverse=True)
G = sparse.csr_matrix((np.ones(len(seg_df)), (b_idx_all, np.arange(len(seg_df)))), shape=(n_books, len(seg_df)))
book_term_docs = (G @ (X_all > 0).astype(np.float32)).toarray()          # books x terms: segments containing the term
books_with_term = (book_term_docs >= MIN_SEGS_PER_BOOK).sum(axis=0)
keep_cols = np.where(books_with_term >= MIN_BOOKS)[0]
print(f"Vocabulary: {len(vocab_all):,} terms -> {len(keep_cols):,} after the shared-across-books filter")

X = normalize(X_all[:, keep_cols])
names = vocab_all[keep_cols]
keep_rows = X.getnnz(axis=1) > 0
if (~keep_rows).any():
    print(f"Dropping {int((~keep_rows).sum())} segments with no remaining vocabulary")
seg_df = seg_df[keep_rows].reset_index(drop=True)
X = X[keep_rows]
B = (X > 0).astype(np.float32).tocsr()                                  # binary matrix for coherence
b_idx = np.unique(seg_df["book"].values, return_inverse=True)[1]
book_codes = np.unique(seg_df["book"].values)
n_books = len(book_codes)
print(f"Model matrix: {X.shape[0]:,} segments x {X.shape[1]:,} terms")

20 books -> a term must appear in >= 2 books (>= 3 segments each)
Vocabulary: 5,687 terms -> 1,897 after the shared-across-books filter
Model matrix: 856 segments x 1,897 terms


## 7. Choose the number of topics

NMF is fitted for each candidate K. **NPMI coherence** measures whether a topic's top words genuinely co-occur in the same passages; **topic diversity** is the share of unique words across all topics' top-10 lists (low = topics are repeating each other). Coherence usually favours small K, so treat the peak as a starting point: if the best K merges things you know are distinct, set `K_FINAL` higher and re-run from the next cell.

In [27]:
def display_terms(row, names, n=8):
    # Top terms for a topic, skipping n-grams that repeat a word already chosen
    # (so "fever" and "fever shed" do not both crowd the label).
    chosen, words = [], set()
    for i in np.argsort(row)[::-1]:
        w = set(names[i].split())
        if w & words:
            continue
        chosen.append(i); words |= w
        if len(chosen) == n:
            break
    return chosen

def npmi_coherence(H, B, names, topn=10):
    # NPMI coherence: do a topic's top words actually co-occur in the same segments?
    # Range -1..1, higher is better. Pairs that share a word (unigram vs. its own bigram)
    # are skipped, otherwise they inflate the score.
    n_docs = B.shape[0]
    df = np.asarray(B.sum(axis=0)).ravel()
    scores = []
    for row in H:
        idx = display_terms(row, names, topn)
        sub = B[:, idx]
        co = (sub.T @ sub).toarray() / n_docs
        p = df[idx] / n_docs
        vals = []
        for i in range(len(idx)):
            for j in range(i + 1, len(idx)):
                pij = co[i, j]
                vals.append(-1.0 if pij <= 0 else np.log(pij / (p[i] * p[j])) / (-np.log(pij)))
        scores.append(np.mean(vals) if vals else 0.0)
    return np.array(scores)

def fit_nmf(X, k):
    m = NMF(n_components=k, init="nndsvda", random_state=SEED, max_iter=500, tol=1e-4)
    W = m.fit_transform(X)
    return m, W

def row_normalise(M):
    s = M.sum(axis=1, keepdims=True)
    out = np.divide(M, s, out=np.full_like(M, np.nan, dtype=float), where=s > 0)
    return out

PALETTE = px.colors.qualitative.Alphabet + px.colors.qualitative.Dark24
def topic_color(t):
    return PALETTE[t % len(PALETTE)]

def show(fig, name):
    fig.write_html(FIG_DIR / f"{name}.html", include_plotlyjs="cdn")
    fig.show()

def plot_topic_words(H, names, labels, n_words=8, ncols=4, title="Top terms per topic"):
    K = H.shape[0]
    nrows = ceil(K / ncols)
    fig = make_subplots(rows=nrows, cols=ncols,
                        subplot_titles=[labels[t][:42] for t in range(K)],
                        horizontal_spacing=0.10, vertical_spacing=0.30 / max(nrows, 1))
    for t in range(K):
        idx = display_terms(H[t], names, n_words)
        fig.add_trace(go.Bar(x=H[t, idx][::-1], y=[names[i] for i in idx][::-1], orientation="h",
                             marker_color=topic_color(t), showlegend=False,
                             hovertemplate="%{y}<extra></extra>"),
                      row=t // ncols + 1, col=t % ncols + 1)
    fig.update_xaxes(showticklabels=False)
    fig.update_yaxes(tickfont=dict(size=10))
    fig.update_annotations(font_size=11)
    fig.update_layout(title=title, height=max(320, 230 * nrows), margin=dict(t=80, b=20))
    return fig

def cluster_order(M, metric="euclidean"):
    if len(M) < 3:
        return list(range(len(M)))
    Z = linkage(pdist(M, metric=metric), method="average", optimal_ordering=True)
    return list(leaves_list(Z))

In [28]:
sweep = []
ks = [k for k in K_RANGE if k < min(X.shape) and k * MIN_SEGS_PER_TOPIC <= X.shape[0]]
if not ks:
    ks = [max(2, min(3, min(X.shape) - 1))]
    print(f"!! Only {X.shape[0]} segments: trying K = {ks[0]} only. Lower MIN_SEGMENT_TOKENS to get more segments.")
for k in tqdm(ks, desc="K sweep"):
    m, _ = fit_nmf(X, k)
    coh = npmi_coherence(m.components_, B, names, COHERENCE_TOPN)
    tops = [tuple(names[display_terms(r, names, COHERENCE_TOPN)]) for r in m.components_]
    diversity = len({w for t in tops for w in t}) / max(1, sum(len(t) for t in tops))
    sweep.append({"k": k, "coherence_mean": coh.mean(), "coherence_min": coh.min(), "diversity": diversity,
                  "reconstruction_err": m.reconstruction_err_})
sweep = pd.DataFrame(sweep)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=sweep.k, y=sweep.coherence_mean, name="mean NPMI coherence", mode="lines+markers"), secondary_y=False)
fig.add_trace(go.Scatter(x=sweep.k, y=sweep.coherence_min, name="worst topic coherence", mode="lines+markers", line=dict(dash="dot")), secondary_y=False)
fig.add_trace(go.Scatter(x=sweep.k, y=sweep.diversity, name="topic diversity", mode="lines+markers"), secondary_y=True)
fig.update_layout(title="Choosing K: coherence and diversity by number of topics", xaxis_title="number of topics (K)", height=420)
fig.update_yaxes(title_text="coherence (NPMI)", secondary_y=False)
fig.update_yaxes(title_text="diversity", range=[0, 1.05], secondary_y=True)
show(fig, "01_k_sweep")

K = int(K_FINAL) if K_FINAL else int(sweep.loc[sweep["coherence_mean"].idxmax(), "k"])
print(f"Using K = {K}" + ("  (set manually)" if K_FINAL else "  (best mean coherence)"))
sweep.round(3)

K sweep:   0%|          | 0/9 [00:00<?, ?it/s]

/Users/sophiebird/anaconda3/envs/dearcanadaenv/lib/python3.11/site-packages/sklearn/decomposition/_nmf.py:1720: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(


Using K = 24  (best mean coherence)


,k,coherence_mean,coherence_min,diversity,reconstruction_err
0,8,0.160,0.089,0.950,27.204
1,12,0.174,0.106,0.925,26.955
2,16,0.176,0.122,0.881,26.755
3,20,0.186,0.109,0.865,26.571
4,24,0.197,0.091,0.883,26.398
5,28,0.185,0.046,0.846,26.255
6,32,0.180,0.026,0.862,26.110
7,36,0.181,0.098,0.831,25.985
8,40,0.181,0.021,0.805,25.860


## 8. Fit the final model & score every entry

In [29]:
nmf, W = fit_nmf(X, K)
H = nmf.components_
Wn = row_normalise(W)                      # segment x topic, each row sums to 1
Wn = np.nan_to_num(Wn)

# Score each individual entry with the fitted model (for entry_topics.csv and the KG join)
X_e = normalize(vec.transform(corpus["lemmas"])[:, keep_cols])
We = row_normalise(nmf.transform(X_e))     # NaN where an entry has no usable vocabulary
topic_cols = [f"topic_{t}" for t in range(K)]
entry_topics = corpus[["uid", "book", "chapter_id", "entry_id", "date", "year", "n_tokens"]].copy()
entry_topics[topic_cols] = We
entry_topics["dominant_topic"] = np.where(np.isnan(We).all(axis=1), -1, np.nanargmax(np.nan_to_num(We, nan=-1), axis=1))
print(f"{int((entry_topics['dominant_topic'] == -1).sum())} entries had no usable vocabulary (dominant_topic = -1)")

27 entries had no usable vocabulary (dominant_topic = -1)


## 9. Name the topics

Auto-labels are the top three distinct terms. Set `USE_LLM_LABELS = True` for Claude-suggested names, descriptions and a `kind` (theme / event / setting / artefact) — the `artefact` flag is useful for spotting topics that are really leaked names or formatting noise. **Treat them as drafts to review**, like your `_reviewed_DRAFT.json` step.

First, the exemplar passages for each topic (highest weight, at most two per book so one book cannot dominate the evidence):

In [30]:
def pick_exemplars(t, n=6, per_book=2):
    order = np.argsort(Wn[:, t])[::-1]
    picked, count = [], {}
    for i in order:
        b = seg_df.loc[i, "book"]
        if count.get(b, 0) >= per_book:
            continue
        picked.append(i); count[b] = count.get(b, 0) + 1
        if len(picked) == n:
            break
    return picked

exemplar_rows = []
for t in range(K):
    for rank, i in enumerate(pick_exemplars(t), 1):
        r = seg_df.loc[i]
        exemplar_rows.append({"topic": t, "rank": rank, "weight": round(float(Wn[i, t]), 3), "book": r["book"],
                              "chapter_id": r["chapter_id"], "first_entry": r["uids"][0].split("|", 1)[1],
                              "year": r["year"], "text": r["text_clean"][:700]})
exemplars = pd.DataFrame(exemplar_rows)

top_terms = {t: [names[i] for i in display_terms(H[t], names, 12)] for t in range(K)}
topic_info = pd.DataFrame({"topic": range(K)})
topic_info["auto_label"] = [" / ".join(top_terms[t][:3]) for t in range(K)]
topic_info["label"] = topic_info["auto_label"]
topic_info["description"] = ""
topic_info["kind"] = ""

if USE_LLM_LABELS:
    payload = [{"topic": t, "top_terms": top_terms[t],
                "exemplars": [{"book": r.book, "year": None if pd.isna(r.year) else int(r.year), "text": r.text[:400]}
                              for r in exemplars[exemplars.topic == t].head(3).itertuples()]}
               for t in range(K)]
    result = llm_label_topics(payload, f"There are {n_books} books in the corpus.")
    for r in result:
        topic_info.loc[topic_info.topic == r["topic"], ["label", "description", "kind"]] = [r["label"], r["description"], r["kind"]]

topic_label = {t: f"T{t}: {topic_info.loc[t, 'label']}" for t in range(K)}
topic_info

,topic,auto_label,label,description,kind
0,0,birthday / book / child,birthday / book / child,,
1,1,settler / indians / buffalo,settler / indians / buffalo,,
2,2,british / soldier / officer,british / soldier / officer,,
3,3,park / camp / car,park / camp / car,,
4,4,france / mademoiselle / marry,france / mademoiselle / marry,,
5,5,ship / deck / captain,ship / deck / captain,,
6,6,baba / money / chinese,baba / money / chinese,,
7,7,camp / bunkhouse / prisoner,camp / bunkhouse / prisoner,,
8,8,mill / fairy / machine,mill / fairy / machine,,
9,9,folk / coloured / guess,folk / coloured / guess,,


## 10. How shared is each topic? Theme, event, or one book's story?

For every topic, computed with **each book weighted equally** (so long books do not dominate):

- `n_books_prominent` — in how many books the topic's share is at least average (`PROMINENCE_FACTOR / K`)
- `evenness` — 1.0 means the topic is spread equally over all books, 0 means it lives in a single book
- `year_median`, `year_iqr` — where the topic's weight sits in historical time, and how wide the middle half of that weight is

Category rules, applied in order: **book-specific** (prominent in ≤ 1 book) → **shared period/event** (≥ 2 books and narrow year span) → **shared theme** (prominent in ≥ 30% of books) → **partially shared**. The thresholds are in the config cell. Books with no readable year are excluded from the year statistics only.

In [31]:
tok_w = seg_df["n_tokens"].values.astype(float)
book_tok_total = np.bincount(b_idx, weights=tok_w)
bw = tok_w / book_tok_total[b_idx]                                   # each book's segment weights sum to 1
years = seg_df["year"].values.astype(float)

# books x topics: share of each book's text belonging to each topic (rows sum to 1)
book_share = np.vstack([np.average(Wn[b_idx == b], axis=0, weights=tok_w[b_idx == b]) for b in range(n_books)])

def wquantile(values, weights, qs):
    o = np.argsort(values); v, w = values[o], weights[o]
    cw = np.cumsum(w) / w.sum()
    return np.interp(qs, cw, v)

rows = []
prom_thr = PROMINENCE_FACTOR / K
for t in range(K):
    prominent = book_share[:, t] >= prom_thr
    p = book_share[:, t] / book_share[:, t].sum()
    evenness = float(-(p[p > 0] * np.log(p[p > 0])).sum() / np.log(n_books)) if n_books > 1 else 1.0
    ok = ~np.isnan(years)
    w_t = (bw * Wn[:, t])[ok]
    if ok.any() and w_t.sum() > 0:
        q25, q50, q75 = wquantile(years[ok], w_t, [0.25, 0.5, 0.75])
    else:
        q25 = q50 = q75 = np.nan
    top_idx = np.argsort(book_share[:, t])[::-1][:4]
    rows.append({"topic": t, "label": topic_label[t], "kind": topic_info.loc[t, "kind"],
                 "top_terms": ", ".join(top_terms[t][:8]),
                 "mean_share": book_share[:, t].mean(), "n_books_prominent": int(prominent.sum()),
                 "evenness": evenness, "year_median": q50, "year_iqr": q75 - q25,
                 "top_books": "; ".join(f"{book_codes[i]} ({book_share[i, t]:.0%})" for i in top_idx)})
topic_summary = pd.DataFrame(rows)

def categorise(r):
    if r.n_books_prominent <= 1:
        return "book-specific"
    if not np.isnan(r.year_iqr) and r.year_iqr <= EVENT_MAX_IQR_YEARS:
        return "shared period / event"
    if r.n_books_prominent >= SHARED_THEME_MIN_BOOK_FRAC * n_books:
        return "shared theme"
    return "partially shared"

topic_summary["category"] = topic_summary.apply(categorise, axis=1)
print(topic_summary["category"].value_counts().to_string())
topic_summary.sort_values(["category", "n_books_prominent"], ascending=[True, False]).drop(columns=["label"]).round(3)

category
partially shared         10
shared theme              7
book-specific             5
shared period / event     2


,topic,kind,top_terms,mean_share,n_books_prominent,evenness,year_median,year_iqr,top_books,category
1,1,,"settler, indians, buffalo, indian, fort, hunt,...",0.027,1,0.545,1816.0,1.380,FitS (33%); AToBD (4%); WofW (3%); BUoL (2%),book-specific
3,3,,"park, camp, car, meeting, radio, road, supper,...",0.022,1,0.588,1941.0,25.000,TA (27%); EftW (3%); PasWide (2%); TFN (1%),book-specific
9,9,,"folk, coloured, guess, horse, cabin, white, wa...",0.033,1,0.665,1863.0,34.000,DRtoF (35%); PasWide (3%); TA (3%); OatmyD (3%),book-specific
13,13,,"métis, tard, government, buffalo, monsieur, st...",0.038,1,0.530,1885.0,0.000,BUoL (51%); FitS (4%); AinUTL (3%); TFN (2%),book-specific
20,20,,"gold, cart, trail, river, pack, fort, foot, field",0.050,1,0.644,1862.0,0.000,AToBD (56%); AinUTL (4%); WofW (4%); FitS (3%),book-specific
12,12,,"war, fight, ride, states, united, fort, allow,...",0.047,5,0.742,1812.0,103.000,WofW (42%); BFfH (6%); TA (6%); NSH (5%),partially shared
18,18,,"boat, river, lake, shore, reach, sailor, water...",0.051,5,0.846,1863.0,83.000,CofOO (19%); NbutC (19%); FitS (15%); DRtoF (9%),partially shared
0,0,,"birthday, book, child, mother, baby, love, sis...",0.027,3,0.873,1916.0,39.246,EftW (10%); BFfH (10%); OatmyD (6%); TFN (3%),partially shared
2,2,,"british, soldier, officer, battle, enemy, woun...",0.033,3,0.610,1759.0,107.000,DofmyC (37%); BUoL (6%); WofW (5%); BFfH (3%),partially shared
5,5,,"ship, deck, captain, sea, sailor, passenger, b...",0.027,3,0.771,1912.0,65.000,TFN (17%); SofS (8%); AinUTL (7%); PasWide (3%),partially shared


## 11. Visualisations

All figures are interactive (hover for detail) and saved as standalone HTML in `output/topics_corpus/figures/`.

### 11a. What each topic is made of

In [32]:
fig = plot_topic_words(H, names, [topic_label[t] for t in range(K)], n_words=8, ncols=4,
                       title="Top terms per topic (bar length = term weight within the topic)")
show(fig, "02_topic_terms")

### 11b. Which books carry which topics

Left: each book's share of each topic. Right: the same numbers as **lift** over the topic's average across books (log₂; red = the book over-uses the topic, blue = under-uses). Books are ordered by the median year of their setting, so period-bound topics should form diagonal bands. Rows with a few strong red cells are book- or period-specific; rows that are evenly pale are broadly shared.

In [33]:
med_year = pd.Series({b: np.nanmedian(years[b_idx == i]) if (~np.isnan(years[b_idx == i])).any() else np.nan
                      for i, b in enumerate(book_codes)})
book_order = list(med_year.sort_values(na_position="last").index)
bi = [list(book_codes).index(b) for b in book_order]
def book_label(b):
    title = BOOK_TITLES.get(b, b)
    return f"{title} ({int(med_year[b])})" if not np.isnan(med_year[b]) else title
xlabels = [book_label(b) for b in book_order]
ylabels = [topic_label[t][:48] for t in range(K)]

M = book_share[bi].T
fig = px.imshow(M, x=xlabels, y=ylabels, aspect="auto", color_continuous_scale="Viridis",
                labels=dict(color="share"), title="Topic share by book")
fig.update_layout(height=max(450, 24 * K + 200), xaxis_tickangle=-60)
show(fig, "03_topic_by_book_share")

lift = np.log2((M + 1e-9) / (M.mean(axis=1, keepdims=True) + 1e-9))
fig = px.imshow(lift, x=xlabels, y=ylabels, aspect="auto", color_continuous_scale="RdBu_r",
                color_continuous_midpoint=0, zmin=-2.5, zmax=2.5, labels=dict(color="log2 lift"),
                title="Topic lift by book (over / under-use relative to the topic's average book)")
fig.update_layout(height=max(450, 24 * K + 200), xaxis_tickangle=-60)
show(fig, "04_topic_by_book_lift")

### 11c. Themes vs. events vs. one-book topics

x = number of books where the topic is prominent; y = how many years the middle half of the topic's weight spans. **Bottom-right** (many books, narrow years) = candidate shared events/periods; **top-right** (many books, wide years) = candidate shared themes; **left** = book-specific. Bubble size = overall share. Topics with no year information are not drawn.

In [34]:
plot_df = topic_summary.dropna(subset=["year_iqr"]).copy()
rng = np.random.default_rng(SEED)
plot_df["x_jit"] = plot_df["n_books_prominent"] + rng.uniform(-0.18, 0.18, len(plot_df))
plot_df["tag"] = "T" + plot_df["topic"].astype(str)
fig = px.scatter(plot_df, x="x_jit", y="year_iqr", size="mean_share", color="category", text="tag",
                 hover_name="label", hover_data={"x_jit": False, "n_books_prominent": True, "year_iqr": ":.1f",
                                                 "year_median": ":.0f", "top_terms": True, "mean_share": ":.3f"},
                 color_discrete_map={"shared theme": "#1f77b4", "shared period / event": "#d62728",
                                     "partially shared": "#7f7f7f", "book-specific": "#bcbd22"},
                 title="How widely is each topic shared, and over how many years?")
fig.update_traces(textposition="top center", marker=dict(line=dict(width=0.5, color="white")))
fig.add_hline(y=EVENT_MAX_IQR_YEARS, line_dash="dot", line_color="grey",
              annotation_text=f"event threshold ({EVENT_MAX_IQR_YEARS} yrs)")
fig.update_layout(xaxis_title="books in which the topic is prominent", yaxis_title="year span of the middle 50% of the topic (IQR, years)", height=560)
show(fig, "05_shared_vs_event")

### 11d. Topics through historical time

Columns are historical periods (the setting year of each passage, not publication date); the label shows how many books contribute. Colour is the topic's **lift** in that period relative to its corpus-wide average. Sparse periods (few books) are noisy, so read the book counts. This is where events show up as isolated hot cells.

In [35]:
ok = ~np.isnan(years)
if ok.sum() == 0:
    print("No year information available: skipping period heatmap. Fill BOOK_START_YEAR in the config.")
    period_matrix = pd.DataFrame()
else:
    bins = (np.floor(years[ok] / BIN_YEARS) * BIN_YEARS).astype(int)
    uniq_bins = np.sort(np.unique(bins))
    P = np.vstack([np.average(Wn[ok][bins == b], axis=0, weights=bw[ok][bins == b]) for b in uniq_bins])  # periods x topics
    books_per_bin = [len(np.unique(b_idx[ok][bins == b])) for b in uniq_bins]
    overall = np.average(Wn[ok], axis=0, weights=bw[ok])
    lift_p = np.log2((P + 1e-9) / (overall + 1e-9)).T
    xl = [f"{b}s (n={n})" for b, n in zip(uniq_bins, books_per_bin)]
    fig = px.imshow(lift_p, x=xl, y=ylabels, aspect="auto", color_continuous_scale="RdBu_r", color_continuous_midpoint=0,
                    zmin=-2.5, zmax=2.5, labels=dict(color="log2 lift"),
                    title=f"Topic lift by {BIN_YEARS}-year period (n = number of books contributing)")
    fig.update_layout(height=max(450, 24 * K + 200), xaxis_tickangle=-45)
    show(fig, "06_topic_by_period")
    period_matrix = pd.DataFrame(P, index=[f"{b}s" for b in uniq_bins], columns=[topic_label[t] for t in range(K)])
    period_matrix.insert(0, "n_books", books_per_bin)

### 11e. How topics relate to each other, and how books relate to each other

Left/top: topics clustered by cosine similarity of their term profiles (blocks = families of related topics, a hint that two topics may be one theme split in two). Bottom: books clustered by Jensen–Shannon distance between their topic profiles (darker = more similar). Books that cluster together tell similar stories in similar words.

In [ ]:
Hn = normalize(H)
order_t = cluster_order(Hn, "cosine")
S = Hn @ Hn.T
fig = px.imshow(S[np.ix_(order_t, order_t)], x=[topic_label[t][:40] for t in order_t], y=[topic_label[t][:40] for t in order_t],
                color_continuous_scale="Magma_r", zmin=0, zmax=1, aspect="equal",
                title="Topic–topic similarity (cosine of term profiles)")
fig.update_layout(height=max(500, 20 * K + 250), xaxis_tickangle=-60)
show(fig, "07_topic_similarity")

if n_books >= 3:
    D = np.array([[jensenshannon(a, b) for b in book_share] for a in book_share])
    order_b = cluster_order(book_share, "jensenshannon")
    lab = [book_label(book_codes[i]) for i in order_b]
    fig = px.imshow(D[np.ix_(order_b, order_b)], x=lab, y=lab, color_continuous_scale="Viridis_r", aspect="equal",
                    labels=dict(color="JS distance"), title="Book–book distance in topic profile")
    fig.update_layout(height=max(500, 18 * n_books + 250), xaxis_tickangle=-60)
    show(fig, "08_book_similarity")
else:
    print("Need at least 3 books for a book-similarity plot.")

## 12. Export

In [ ]:
'''
topic_summary.to_csv(OUT_DIR / "topic_summary.csv", index=False)
pd.DataFrame([{"topic": t, "rank": r + 1, "term": names[i], "weight": H[t, i]}
              for t in range(K) for r, i in enumerate(display_terms(H[t], names, 25))]).to_csv(OUT_DIR / "topic_terms.csv", index=False)
bt = pd.DataFrame(book_share[bi], index=book_order, columns=[topic_label[t] for t in range(K)])
bt.insert(0, "median_year", med_year[book_order].values)
bt.to_csv(OUT_DIR / "book_topic_matrix.csv")
if len(period_matrix):
    period_matrix.to_csv(OUT_DIR / "topic_period_matrix.csv")
exemplars.to_csv(OUT_DIR / "exemplars.csv", index=False)
topic_info.to_csv(OUT_DIR / "topic_labels.csv", index=False)
entry_topics.to_csv(OUT_DIR / "entry_topics.csv", index=False)
print("Wrote to", OUT_DIR.resolve())
for f in sorted(OUT_DIR.glob("*.csv")):
    print("  ", f.name)
'''

## 13. Reading the results (and what to check)

1. **Read `exemplars.csv` before trusting any label.** Six passages per topic, at most two per book. If a topic's passages are all from one book, it is a book-specific topic regardless of what the categories say.
2. **Look for `artefact` topics** (or topics whose top terms are one family's kin words, a ship's name, a recurring phrase). Add the offending words to `EXTRA_STOPWORDS`, delete `cache/` only if you changed the spaCy settings, and re-run.
3. **Tune `MIN_BOOK_FRACTION` deliberately.** It trades *breadth* (universal themes) against *coverage of events* that only a few books share. Running twice, once at 0.08 and once at 0.20, and comparing which topics survive, is itself informative.
4. **K is a judgement call.** Compare the sweep's best K against a K 4–8 higher; real sub-themes often split off (e.g. one "illness" topic becomes "ship fever" and "grief").
5. **"Event" ≠ confirmed event.** A narrow year span plus several books suggests a shared historical moment; confirming it is a historical task. The `top_books` column tells you where to look.
6. **`entry_topics.csv` joins to your KG** on `book` + `entry_id`, so topic weights can become properties on `Entry` nodes or edges to a new `Topic` node.